# Extract AI structure scores

Get the AF2 pLDDT and DSSP RSA scores from .pdb files

In [10]:
import os
import sys
import gzip
import glob
import warnings
from pathlib import Path
from functools import partial
from multiprocessing import Pool, cpu_count
from typing import List, Dict, Tuple, Optional, Union
import numpy as np
import pandas as pd
import polars as pl
import pyarrow as pa
import pyarrow.parquet as pq
from Bio.PDB import PDBParser
from io import StringIO

sys.path.insert(1, "scripts/")
import extract_plddt_scores as get_plddt
import calculate_rsa_scores as get_rsa

## pLDDT & RSA

### pLDDT

In [10]:
pdb_file = "/home/shenrunx/igvf/varchamp/2025_laval_submitted/4_compare_AI_scores/1_inputs/raw_inputs/UP000005640_9606_HUMAN_v4/AF-X6R8D5-F1-model_v4.pdb.gz"

# Step 3: Extract pLDDT Scores
def extract_plddt(pdb_file):
    """Parse AlphaFold output for per-residue pLDDT scores from a .gz PDB file"""
    with gzip.open(pdb_file, 'rt') as f:
        pdb_content = f.read()
    parser = PDBParser()
    structure = parser.get_structure("AF", StringIO(pdb_content))
    return [(residue.get_id()[1], residue.get_resname(), residue["CA"].bfactor) for residue in structure.get_residues()]

plddt_scores = extract_plddt(pdb_file=pdb_file)
plddt_scores[:10]

[(1, 'MET', 43.9),
 (2, 'GLY', 47.84),
 (3, 'ARG', 53.25),
 (4, 'LYS', 47.35),
 (5, 'GLU', 58.27),
 (6, 'HIS', 42.5),
 (7, 'GLU', 45.51),
 (8, 'SER', 52.87),
 (9, 'PRO', 45.72),
 (10, 'SER', 50.8)]

In [13]:
uniprot_id = "X6R8D5"
pdb_lines = []
with gzip.open(pdb_file, 'rt') as f:
    for i, line in enumerate(f):
        line = line[:-1].split()
        if line[0] == 'ATOM':
            if len(line) == 12: #solve issue for lines with merged columns
                pos = int(line[5])
                res = line[3]
                plddt = line[-2]
                row = [uniprot_id, pdb_file, pos, res, float(plddt)]
                pdb_lines.append(row)

# with open (pdb_file) as f:
#     for i, line in enumerate(f):
#         line = line[:-1].split()
#         if line[0] == 'ATOM':
#             if len(line) == 12: #solve issue for lines with merged columns
#                 pos = int(line[5])
#                 res = line[3]
#                 plddt = line[-2]
#                 row = [uniprot_id,file,pos,res,float(plddt)]
#                 pdb_lines.append(row)

In [17]:
parquet_dir = "../3_outputs/processed_data/plddt_scores"
parquet_files = [f"{parquet_dir}/{par_f}" for par_f in os.listdir(parquet_dir)]
parquet_files

# 1. Get all scores for a specific protein
df = get_plddt.query_plddt(parquet_files, pdb_ids=["AF-X6R8D5-F1-model_v4"])
df

,pdb_id,chain_id,residue_num,residue_name,plddt
0,AF-X6R8D5-F1-model_v4,A,1,MET,43.90
1,AF-X6R8D5-F1-model_v4,A,2,GLY,47.84
2,AF-X6R8D5-F1-model_v4,A,3,ARG,53.25
3,AF-X6R8D5-F1-model_v4,A,4,LYS,47.35
4,AF-X6R8D5-F1-model_v4,A,5,GLU,58.27
...,...,...,...,...,...
122,AF-X6R8D5-F1-model_v4,A,123,VAL,83.56
123,AF-X6R8D5-F1-model_v4,A,124,LEU,81.48
124,AF-X6R8D5-F1-model_v4,A,125,TRP,80.66
125,AF-X6R8D5-F1-model_v4,A,126,ARG,79.65


### RSA scores

In [18]:
parquet_files = [par_f for par_f in glob.glob("../3_outputs/processed_data/rsa_scores")]
parquet_files

pdb1 = "AF-X6R8D5-F1-model_v4"
# pdb1 = "AF-Q5VUB5-F1-model_v4"

df_6lzg = get_rsa.get_structure_rsa(parquet_files, pdb1)
print(f"Retrieved {len(df_6lzg)} residues for {pdb1}")
df_6lzg

# 2. Get RSA for a specific residue
# rsa = get_rsa.get_residue_rsa(parquet_files, pdb1, "A", 124)
# rsa = get_rsa.get_residue_rsa(parquet_files, pdb1, "A", 124)
# print(f"RSA for {pdb1} chain A residue 42: {rsa}%")

# # 3. Advanced query - all residues with RSA > 50%
# all_data = get_rsa.query_rsa_data(parquet_files)
# exposed_residues = all_data[all_data['rsa'] > 50.0]
# print(f"Found {len(exposed_residues)} exposed residues (RSA > 50%)")

Retrieved 127 residues for AF-X6R8D5-F1-model_v4


,pdb_id,chain_id,residue_num,residue_name,rsa
0,AF-X6R8D5-F1-model_v4,A,1,MET,1.000000
1,AF-X6R8D5-F1-model_v4,A,2,GLY,0.845238
2,AF-X6R8D5-F1-model_v4,A,3,ARG,1.000000
3,AF-X6R8D5-F1-model_v4,A,4,LYS,0.839024
4,AF-X6R8D5-F1-model_v4,A,5,GLU,1.000000
...,...,...,...,...,...
122,AF-X6R8D5-F1-model_v4,A,123,VAL,0.739437
123,AF-X6R8D5-F1-model_v4,A,124,LEU,0.676829
124,AF-X6R8D5-F1-model_v4,A,125,TRP,0.744493
125,AF-X6R8D5-F1-model_v4,A,126,ARG,0.814516


## dbNSFP

In [3]:
dbnsfp_df = pl.scan_parquet(f"../3_outputs/processed_data/dbnsfp/dbNSFP5.0a_variant.clin_var_re-annot_pdb_variants.parquet")
dbnsfp_df = dbnsfp_df.collect()

In [16]:
dbnsfp_df.select(pl.col("clinvar_id", "aaref_3", "aaref", "Uniprot_acc", "aapos", "clinvar_aa_change", "HGVSp_snpEff", "HGVSp_VEP")).head()

clinvar_id,aaref_3,aaref,Uniprot_acc,aapos,clinvar_aa_change,HGVSp_snpEff,HGVSp_VEP
str,str,str,str,str,str,str,str
"""2361281""","""Ser""","""S""","""Q86YR6""","""396""","""Ser396Ile""","""p.Ser396Ile""","""p.Ser396Ile"""
"""2599176""","""Val""","""V""","""Q86YR6""","""415""","""Val415Phe""","""p.Val415Phe""","""p.Val415Phe"""
"""2486768""","""Val""","""V""","""Q86YR6""","""427""","""Val427Met""","""p.Val427Met""","""p.Val427Met"""
"""2277922""","""Arg""","""R""","""Q86YR6""","""453""","""Arg453Ser""","""p.Arg453Ser""","""p.Arg453Ser"""
"""3308991""","""Thr""","""T""","""Q86YR6""","""495""","""Thr495Ser""","""p.Thr495Ser""","""p.Thr495Ser"""


In [21]:
PLDDT_DIR = "../3_outputs/processed_data/plddt_scores"

def get_plddt_score(pdb_id, aapos, aaref, parquet_files_plddt=[f"{PLDDT_DIR}/{par_f}" for par_f in os.listdir(PLDDT_DIR) if par_f.endswith('.parquet')]):
    """Get pLDDT score for a specific residue."""
    df = get_plddt.query_plddt(parquet_files_plddt, pdb_ids=[f"AF-{pdb_id}-F1-model_v4"])
    if df is None or df.empty:
        return np.nan

    print(df)
    print(df.loc[int(aapos),])

    try:
        if int(aapos)-1 in df.index and df.loc[int(aapos)-1, "residue_name"] == aaref:
            return df.loc[int(aapos)-1, "plddt"]
    except:
        pass
    return np.nan

get_plddt_score("Q86YR6", "396", "SER")

                    pdb_id chain_id  residue_num residue_name  plddt
0    AF-Q86YR6-F1-model_v4        A            1          MET  38.20
1    AF-Q86YR6-F1-model_v4        A            2          VAL  39.78
2    AF-Q86YR6-F1-model_v4        A            3          ALA  37.65
3    AF-Q86YR6-F1-model_v4        A            4          GLU  37.11
4    AF-Q86YR6-F1-model_v4        A            5          VAL  38.30
..                     ...      ...          ...          ...    ...
579  AF-Q86YR6-F1-model_v4        A          580          THR  82.04
580  AF-Q86YR6-F1-model_v4        A          581          LYS  81.94
581  AF-Q86YR6-F1-model_v4        A          582          THR  78.21
582  AF-Q86YR6-F1-model_v4        A          583          ASN  65.15
583  AF-Q86YR6-F1-model_v4        A          584          ILE  54.24

[584 rows x 5 columns]
pdb_id          AF-Q86YR6-F1-model_v4
chain_id                            A
residue_num                       397
residue_name                      

32.34

In [ ]:
dbnsfp_df_sample = dbnsfp_df.collect().sample(n=10000)
print(dbnsfp_df_sample)
# dbnsfp_df_sample.write_parquet("../3_outputs/processed_data/dbnsfp/dbNSFP5.0a_variant.clin_var_re-annot_dedup_variants_10k.parquet")

### Long protein sequences problem

Some proteins have very long sequences, where their structures could not be inferred by AF2 directly as a whole, and EMBL breaks their sequences into chunks for structure inferences separately.

In [25]:
PLDDT_DIR = "../3_outputs/processed_data/plddt_scores"
RSA_DIR = "../3_outputs/processed_data/rsa_scores"

# Get parquet files for pLDDT and RSA scores
parquet_files_plddt = [f"{PLDDT_DIR}/{par_f}" for par_f in os.listdir(PLDDT_DIR) if par_f.endswith('.parquet')]
parquet_files_rsa = [f"{RSA_DIR}/{par_f}" for par_f in os.listdir(RSA_DIR) if par_f.endswith('.parquet')]

In [26]:
pdb_id = "Q8WZ42"
df = get_plddt.query_plddt(parquet_files_plddt, pdb_ids=[f"AF-{pdb_id}-F2-model_v4"])

# df[df["residue_num"]==311]
# df[df["residue_num"]<=311].tail(10)

,pdb_id,chain_id,residue_num,residue_name,plddt
0,AF-Q8WZ42-F2-model_v4,A,1,ALA,39.30
1,AF-Q8WZ42-F2-model_v4,A,2,LYS,38.72
2,AF-Q8WZ42-F2-model_v4,A,3,LYS,42.24
3,AF-Q8WZ42-F2-model_v4,A,4,THR,41.45
4,AF-Q8WZ42-F2-model_v4,A,5,LYS,40.27
...,...,...,...,...,...
1395,AF-Q8WZ42-F2-model_v4,A,1396,ILE,82.81
1396,AF-Q8WZ42-F2-model_v4,A,1397,VAL,73.36
1397,AF-Q8WZ42-F2-model_v4,A,1398,PRO,67.68
1398,AF-Q8WZ42-F2-model_v4,A,1399,HIS,59.86


In [35]:
df = get_plddt.query_plddt(parquet_files_plddt, pdb_ids=[f"AF-{pdb_id}-F1-model_v4"])
df

,pdb_id,chain_id,residue_num,residue_name,plddt
0,AF-Q8WZ42-F1-model_v4,A,1,MET,56.65
1,AF-Q8WZ42-F1-model_v4,A,2,THR,78.09
2,AF-Q8WZ42-F1-model_v4,A,3,THR,86.55
3,AF-Q8WZ42-F1-model_v4,A,4,GLN,88.68
4,AF-Q8WZ42-F1-model_v4,A,5,ALA,92.21
...,...,...,...,...,...
1395,AF-Q8WZ42-F1-model_v4,A,1396,LEU,35.55
1396,AF-Q8WZ42-F1-model_v4,A,1397,GLU,32.15
1397,AF-Q8WZ42-F1-model_v4,A,1398,PRO,41.15
1398,AF-Q8WZ42-F1-model_v4,A,1399,VAL,36.35


In [31]:
df_clinvar_sampled.filter(pl.col("Uniprot_acc")==pdb_id)

#chr,pos(1-based),ref,alt,aaref,aaalt,rs_dbSNP,hg19_chr,hg19_pos(1-based),hg18_chr,hg18_pos(1-based),aapos,genename,Ensembl_geneid,Ensembl_transcriptid,Ensembl_proteinid,Uniprot_acc,Uniprot_entry,HGVSc_snpEff,HGVSp_snpEff,HGVSc_VEP,HGVSp_VEP,APPRIS,GENCODE_basic,TSL,VEP_canonical,MANE,cds_strand,refcodon,codonpos,codon_degeneracy,Ancestral_allele,AltaiNeandertal,Denisova,VindijiaNeandertal,ChagyrskayaNeandertal,SIFT_score,…,nsv/esv (dbVar),RCVaccession,PhenotypeIDS,PhenotypeList,Origin,OriginSimple,Assembly,ChromosomeAccession,Chromosome,Start,Stop,Cytogenetic,ReviewStatus,NumberSubmitters,Guidelines,TestedInGTR,OtherIDs,SubmitterCategories,VariationID,PositionVCF,ReferenceAlleleVCF,AlternateAlleleVCF,SomaticClinicalImpact,SomaticClinicalImpactLastEvaluated,ReviewStatusClinicalImpact,Oncogenicity,OncogenicityLastEvaluated,ReviewStatusOncogenicity,SCVsForAggregateGermlineClassification,SCVsForAggregateSomaticClinicalImpact,SCVsForAggregateOncogenicityClassification,clinvar_nt_change,clinvar_aa_change,SPDI,RefSeq_mRNA,StarStatus,clinvar_clnsig_clean
str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,…,str,str,str,str,str,str,cat,str,str,i64,i64,str,cat,i64,str,str,str,i64,i64,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i32,cat
"""2""",178719695,"""T""","""C""","""S""","""G""","""rs1553905723""","""2""","""179584422""","""2""","""179292667""","""7616""","""TTN""","""ENSG00000155657""","""ENST00000591111""","""ENSP00000465570""","""Q8WZ42""","""TITIN_HUMAN""","""c.22846A>G""","""p.Ser7616Gly""","""c.22846A>G""","""p.Ser7616Gly""",""".""","""Y""","""5""",""".""",""".""","""-""","""AGC""","""1""","""0""","""T""","""T/T""","""T/T""","""T/T""","""T/T""",""".""",…,"""-""","""RCV000549703""","""MONDO:MONDO:0011400,MedGen:C18…","""Dilated cardiomyopathy 1G;Auto…","""germline""","""germline""","""GRCh38""","""NC_000002.12""","""2""",178719695,178719695,"""2q31.2""","""criteria provided, single subm…",1,"""ACMG2021,ACMG2022""","""N""","""ClinGen:CA349505924""",2,466925,178719695,"""T""","""C""","""-""","""-""","""-""","""-""","""-""","""-""","""SCV000642843""","""-""","""-""","""23797A>G ""","""Ser7933Gly""","""NC_000002.12:178719695:T:C""","""NM_001267550.2""",1,"""4_VUS"""
"""2""",178564282,"""C""","""T""","""V""","""I""","""rs746222222""","""2""","""179429009""","""2""","""179137255""","""25643""","""TTN""","""ENSG00000155657""","""ENST00000591111""","""ENSP00000465570""","""Q8WZ42""","""TITIN_HUMAN""","""c.76927G>A""","""p.Val25643Ile""","""c.76927G>A""","""p.Val25643Ile""",""".""","""Y""","""5""",""".""",""".""","""-""","""GTC""","""1""","""0""","""C""","""C/C""","""C/C""","""C/C""","""C/C""",""".""",…,"""-""","""RCV000216643|RCV000727540""","""MedGen:CN169374|MedGen:CN51720…","""not specified|not provided""","""germline""","""germline""","""GRCh38""","""NC_000002.12""","""2""",178564282,178564282,"""2q31.2""","""criteria provided, conflicting…",2,"""ACMG2021,ACMG2022""","""N""","""ClinGen:CA1989144""",2,228149,178564282,"""C""","""T""","""-""","""-""","""-""","""-""","""-""","""-""","""SCV000271088|SCV000709588""","""-""","""-""","""81850G>A ""","""Val27284Ile""","""NC_000002.12:178564282:C:T""","""NM_001267550.2""",1,"""3_Conflicting"""
"""2""",178565406,"""G""","""A""","""P""","""L""","""rs1553581878""","""2""","""179430133""","""2""","""179138379""","""25268""","""TTN""","""ENSG00000155657""","""ENST00000591111""","""ENSP00000465570""","""Q8WZ42""","""TITIN_HUMAN""","""c.75803C>T""","""p.Pro25268Leu""","""c.75803C>T""","""p.Pro25268Leu""",""".""","""Y""","""5""",""".""",""".""","""-""","""CCT""","""2""","""0""","""G""","""G/G""","""G/G""","""G/G""","""G/G""",""".""",…,"""-""","""RCV000559636""","""MONDO:MONDO:0011400,MedGen:C18…","""Dilated cardiomyopathy 1G;Auto…","""germline""","""germline""","""GRCh38""","""NC_000002.12""","""2""",178565406,178565406,"""2q31.2""","""criteria provided, single subm…",1,"""ACMG2021,

In [12]:
accid_df.collect().head()

column_1,column_2,column_3,column_4,column_5
str,i64,i64,str,i64
"""A0A2I1YHU5""",1,933,"""AF-A0A2I1YHU5-F1""",4
"""A0A5H2Z360""",1,342,"""AF-A0A5H2Z360-F1""",4
"""A0A6L5B7P9""",1,275,"""AF-A0A6L5B7P9-F1""",4
"""A0A1G9ZVL8""",1,268,"""AF-A0A1G9ZVL8-F1""",4
"""A0A7G6RQS6""",1,316,"""AF-A0A7G6RQS6-F1""",4


In [14]:
accid_df.filter(pl.col("column_4").str.contains("Q8WZ42")).collect()

column_1,column_2,column_3,column_4,column_5
str,i64,i64,str,i64
"""A0A0Q8WZ42""",1,458,"""AF-A0A0Q8WZ42-F1""",4
"""A0A3Q8WZ42""",1,416,"""AF-A0A3Q8WZ42-F1""",4


In [53]:
df_acc_id.filter(pl.col("uniprot_id").str.contains("Q8WZ42")).collect()

uniprot_id,first_res_idx,last_res_idx,af_db_identifier,latest_version
str,i64,i64,str,i64
"""A0A0Q8WZ42""",1,458,"""AF-A0A0Q8WZ42-F1""",4
"""A0A3Q8WZ42""",1,416,"""AF-A0A3Q8WZ42-F1""",4


In [44]:
import polars as pl
import re
from typing import Optional, Tuple

def find_af_file(parquet_path: str, uniprot_id: str, residue_pos: int) -> Optional[Tuple[str, int]]:
    """
    Find corresponding AF file and subfile index for a residue position
    [Uses binary search on sorted intervals for O(log n) performance]
    """
    # Load only relevant partitions
    df = pl.scan_parquet(parquet_path)
    
    result = df.filter(
        (pl.col("uniprot_id") == uniprot_id) &
        (pl.col("first_res_idx") <= residue_pos) &
        (pl.col("last_res_idx") >= residue_pos)
    ).collect()
    
    if result.height == 1:
        row = result.row(0, named=True)
        return (row["af_db_identifier"], row["subfile_index"])
    
    print(result)
    return (None, None)


def find_af_entry(parquet_path: str, uniprot_id: str, residue_pos: int) -> Optional[dict]:
    """
    Find complete AF file entry with residue range checking
    Returns full row as dictionary if found
    """
    df = pl.scan_parquet(parquet_path)
    result = df.filter(
        (pl.col("uniprot_id") == uniprot_id) &
        (pl.col("first_res_idx") <= residue_pos) &
        (pl.col("last_res_idx") >= residue_pos)
    ).collect()
    
    return result.row(0, named=True) if result.height == 1 else None


def calculate_local_position(af_entry: dict, residue_pos: int) -> int:
    """
    Calculate position within the subfile using 1-based indexing
    Formula: local_pos = residue_pos - first_res_idx + 1
    """
    return residue_pos - af_entry["first_res_idx"] + 1


# Integrated usage example
def get_af_file_and_position(parquet_path: str, uniprot_id: str, residue_pos: int):
    entry = find_af_entry(parquet_path, uniprot_id, residue_pos)
    if not entry:
        return None
        
    return {
        "af_id": entry["af_db_identifier"],
        "subfile_index": entry["subfile_index"],
        "local_position": calculate_local_position(entry, residue_pos),
        "global_range": (entry["first_res_idx"], entry["last_res_idx"])
    }


def get_plddt_score(pdb_id: str, aa_pos: int, aaref: str, 
                    parquet_files_plddt: str, accession_parquet: str, 
                    plddt_module) -> float:
    """Enhanced pLDDT lookup with dynamic AF model selection and position conversion"""
    # 1. Find correct AlphaFold file and local position
    af_entry = find_af_entry(accession_parquet, pdb_id, aa_pos)
    
    if not af_entry:
        return None
    
    # 2. Convert global position to subfile-local index
    local_pos = aa_pos - af_entry["first_res_idx"] + 1  # [1](@ref)
    
    # 3. Query pLDDT data from correct AF model
    df = plddt_module.query_plddt(
        parquet_files_plddt, 
        pdb_ids=[af_entry["af_db_identifier"]]
    )
    # 4. Validate and return score
    try:
        if (local_pos in df.index and 
            df.loc[local_pos, "residue_name"] == aaref):
            return df.loc[local_pos, "plddt"]
    except (KeyError, TypeError):
        pass
    
    return None


# Usage example

# print(find_af_file("../3_outputs/processed_data/af_id_mapping/af_mapping.parquet", 
#                    "A0A2I1YHU5", 500))
# Output: ("AF-A0A2I1YHU5-F1", 1)

find_af_file("../3_outputs/processed_data/af_id_mapping/af_mapping.parquet", pdb_id, 1400)

shape: (0, 6)
┌────────────┬───────────────┬──────────────┬──────────────────┬────────────────┬───────────────┐
│ uniprot_id ┆ first_res_idx ┆ last_res_idx ┆ af_db_identifier ┆ latest_version ┆ subfile_index │
│ ---        ┆ ---           ┆ ---          ┆ ---              ┆ ---            ┆ ---           │
│ str        ┆ i64           ┆ i64          ┆ str              ┆ i64            ┆ str           │
╞════════════╪═══════════════╪══════════════╪══════════════════╪════════════════╪═══════════════╡
└────────────┴───────────────┴──────────────┴──────────────────┴────────────────┴───────────────┘


(None, None)

In [22]:
#### @Deprecated testing

# parquet_dir_plddt = "../3_outputs/processed_data/plddt_scores"
# parquet_files_plddt = [f"{parquet_dir_plddt}/{par_f}" for par_f in os.listdir(parquet_dir_plddt)]

# def get_plddt_score(pdb_id, aapos, aaref, parquet_files_plddt=parquet_files_plddt):
#     # 1. Get all scores for a specific protein
#     df = get_plddt.query_plddt(parquet_files_plddt, pdb_ids=[f"AF-{pdb_id}-F1-model_v4"])
#     if df is None or df.empty:
#         return np.nan
    
#     try:
#         if int(aapos) in df.index and df.loc[int(aapos), "residue_name"] == aaref:
#             return df.loc[int(aapos), "plddt"]
#     except:
#         pass
#     return np.nan
    

# parquet_dir_rsa = "../3_outputs/processed_data/rsa_scores"
# parquet_files_rsa = [f"{parquet_dir_rsa}/{par_f}" for par_f in os.listdir(parquet_dir_rsa)]

# def get_rsa_score(pdb_id, aapos, aaref, parquet_files_rsa=parquet_files_rsa):
#     # 1. Get all scores for a specific protein
#     df = get_rsa.get_structure_rsa(parquet_files_rsa, pdb_id=f"AF-{pdb_id}-F1-model_v4")
#     if df is None or df.empty:
#         return np.nan
    
#     try:
#         if int(aapos) in df.index and df.loc[int(aapos), "residue_name"] == aaref:
#             return df.loc[int(aapos), "rsa"]
#     except:
#         pass
#     return np.nan
    
# ## testing uniprot score mapping
# df_transformed = (
#     ldf.with_columns(
#         pl.struct(pl.col(["Uniprot_acc", "aapos", "aaref_3"]))
#           .map_elements(
#               lambda x: get_plddt_score(
#                   pdb_id=x["Uniprot_acc"].split('-')[0],
#                   aapos=int(x["aapos"]),
#                   aaref=x["aaref_3"].upper(),
#                   parquet_files_plddt=parquet_files_plddt
#               ),
#               return_dtype=pl.Float64
#           )
#           .alias("plddt"),
#         pl.struct(pl.col(["Uniprot_acc", "aapos", "aaref_3"]))
#           .map_elements(
#               lambda x: get_rsa_score(
#                   pdb_id=x["Uniprot_acc"].split('-')[0],
#                   aapos=int(x["aapos"]),
#                   aaref=x["aaref_3"].upper(),
#                   parquet_files_rsa=parquet_files_rsa
#               ),
#               return_dtype=pl.Float64
#           )
#           .alias("rsa"),
#         # pl.struct(pl.col(["Ensembl_proteinid", "aapos", "aaref_3", "aaalt"]))
#         #   .map_elements(
#         #       lambda x: get_rsa_score(
#         #           prot_id=x["Ensembl_proteinid"],
#         #           aapos=int(x["aapos"]),
#         #           aaref=x["aaref_3"],
#         #           aa=x["aaalt"]
#         #       ),
#         #       return_dtype=pl.Float64
#         #   )
#         #   .alias("provean_alt")
#     )
# )

# df_transformed

In [43]:
af_accession_id = pl.scan_parquet("../3_outputs/processed_data/af_id_mapping/af_mapping.parquet")
af_accession_id.collect()

uniprot_id,first_res_idx,last_res_idx,af_db_identifier,latest_version,subfile_index
str,i64,i64,str,i64,str
"""A0A000""",1,394,"""AF-A0A000-F1""",4,"""1"""
"""A0A001""",1,591,"""AF-A0A001-F1""",4,"""1"""
"""A0A002""",1,564,"""AF-A0A002-F1""",4,"""1"""
"""A0A003""",1,340,"""AF-A0A003-F1""",4,"""1"""
"""A0A004""",1,645,"""AF-A0A004-F1""",4,"""1"""
…,…,…,…,…,…
"""Z9JZ66""",1,610,"""AF-Z9JZ66-F1""",4,"""1"""
"""Z9JZ70""",1,309,"""AF-Z9JZ70-F1""",4,"""1"""
"""Z9JZ75""",1,343,"""AF-Z9JZ75-F1""",4,"""1"""


In [48]:
af_accession_id.filter(pl.col("uniprot_id").str.contains(pdb_id)).collect()

uniprot_id,first_res_idx,last_res_idx,af_db_identifier,latest_version,subfile_index
str,i64,i64,str,i64,str
"""A0A0Q8WZ42""",1,458,"""AF-A0A0Q8WZ42-F1""",4,"""1"""
"""A0A3Q8WZ42""",1,416,"""AF-A0A3Q8WZ42-F1""",4,"""1"""
